# patgen — full offline pipeline notebook

This notebook runs the complete patgen pipeline on any CSV file:

1. load raw messages and auto-detect the text column
2. normalize, tokenize, and mask entities
3. learn templates with the Drain-style clusterer
4. benchmark production matching throughput
5. profile learning and matching with `cProfile`
6. inspect coverage, top templates, and example matches

**To use your own data** change `CSV_PATH` in the cell below. The container image
in `.devcontainer/` already has Python, Jupyter, and `patgen` installed, so no
network is needed while the notebook runs.

In [1]:
# --- CONFIGURATION: edit these lines for your CSV ------------------------
CSV_PATH = "examples/sms_sample.csv"  # relative to this notebook's directory
TEXT_COLUMN = None                    # e.g. "message"; None = auto-detect
LEXICON = None                        # e.g. "finance" or a path/to/lexicon.json

# Learning knobs
SIM = 0.5
DEPTH = 4
MAX_CHILDREN = 128
MIN_SUPPORT = 1
DROP_DEGENERATE = True
MAX_TEXT_SLOT = 6
REFINE_RATIO = 0.5
LIMIT = None

# Outputs
MODEL_OUT = "model.json"


In [2]:
import cProfile
import pstats
import random
import time
from collections import Counter
from pathlib import Path

import patgen
from patgen import LearnConfig, TemplateMatcher, learn_templates, list_lexicons
from patgen.io_csv import read_texts
from patgen.report import coverage_report, render_library


In [3]:
csv_path = Path(CSV_PATH)
if not csv_path.is_absolute():
    csv_path = Path.cwd() / csv_path
    # Handle running from inside the notebooks/ directory
    if not csv_path.exists() and Path.cwd().name == "notebooks":
        csv_path = Path.cwd().parent / CSV_PATH

print("CSV path:", csv_path.resolve())
print("Working directory:", Path.cwd().resolve())
print("patgen version:", patgen.__version__)
print("Available bundled lexicons:", list_lexicons())


CSV path: /workspaces/patternExtract/examples/sms_sample.csv
Working directory: /workspaces/patternExtract/notebooks
patgen version: 0.1.0
Available bundled lexicons: ['finance']


In [4]:
messages = list(read_texts([str(csv_path)], column=TEXT_COLUMN, limit=LIMIT))
print(f"Loaded {len(messages):,} non-empty messages")
print("\n--- first 5 messages ---")
for m in messages[:5]:
    print(m)


Loaded 5,000 non-empty messages

--- first 5 messages ---
سحب  نقدي 35,537.14 AED من الحساب ***138 بتاريخ 11/12/2024 الرصيد المتاح 24,190.80
‏Ticket 633021001 updated. Agent ahmed k. replied: issue resolved on our end‎
‏سحب  نقدي 37,955.74 ر.س من الحساب ***693 بتاريخ 2024-02-25 الرصيد المتاح 481,011.44‎
عملية شراء بمبلغ 82,513.88 ر.س لدى بنده بالبطاقة xxxx3243 بتاريخ 2024-05-24 الرصيد المتاح 248,278.52
ALERT host db-master cpu 73% at 11:11 service sms alerts status confirmed


## 1. Learn templates

`LearnConfig` controls the Drain tree and the optional lexicon.

In [5]:
config = LearnConfig(
    sim_threshold=SIM,
    depth=DEPTH,
    max_children=MAX_CHILDREN,
    min_support=MIN_SUPPORT,
    drop_degenerate=DROP_DEGENERATE,
    max_text_slot_tokens=MAX_TEXT_SLOT,
    refine_ratio=REFINE_RATIO,
    lexicon=LEXICON,
)

t0 = time.perf_counter()
library = learn_templates(messages, config=config)
learn_seconds = time.perf_counter() - t0

print(f"Learned {len(library.templates)} templates from {len(messages):,} messages in {learn_seconds:.2f}s")
if MODEL_OUT:
    out_path = Path(MODEL_OUT)
    if not out_path.is_absolute():
        out_path = Path.cwd() / out_path
    library.save(str(out_path))
    print("Saved model to:", out_path.resolve())


Learned 66 templates from 5,000 messages in 0.82s
Saved model to: /workspaces/patternExtract/notebooks/model.json


## 2. Coverage report

In [6]:
cov = coverage_report(library, messages)
print(f"Matched: {cov['matched']:,} / {cov['messages_scored']:,}")
print(f"Coverage: {cov['coverage']:.1%}")
print("\nTop slots by frequency:")
for slot, n in Counter(cov['slot_frequency']).most_common(15):
    print(f"  {slot}: {n:,}")
print("\nUnmatched sample count:", len(cov['unmatched_samples']))
for m in cov['unmatched_samples'][:5]:
    print("  -", m)


Matched: 4,818 / 5,000
Coverage: 96.4%

Top slots by frequency:
  currency: 2,082
  text: 1,480
  date: 1,384
  account: 621
  الحساب: 603
  time: 587
  currency_2: 557
  بمبلغ: 458
  balance: 406
  الي: 394
  ticket: 238
  salary: 235
  reminder: 227
  card: 227
  code: 226

Unmatched sample count: 25
  - Booking 774237069 confirmed. Flight SV102 from cairo to الرياض on 04/04/2024 at 20:56. 
  - Bill payment SAR 60,162.87 to نون succeeded. Fee SAR 79.02. Ref 896621877 thank you
  - Dear customer, service الحوالات الفورية was activated on account ***945. Call 9
  - رمز Ø§Ù„ØªØ­Ù‚Ù‚ Ø§Ù„Ø®Ø§Øµ Ø¨Ùƒ Ù‡Ùˆ 627216 ØµØ§Ù„Ø­ لمدة 5 دقائق لا تشاركه مع احد
  - Dear customer, service المدفوعات الدولية was activated on account ***687. Call 9


## 3. Top learned templates

In [7]:
print("Top templates by message count:\n")
for t in library.templates[:15]:
    print(f"{t.template_id} ({t.count:,} msgs): {t.text}")


Top templates by message count:

t00002 (240 msgs): عمليه شراء بمبلغ <AMOUNT:بمبلغ> <CURRENCY:currency> لدي <TEXT:text>?
t00044 (227 msgs): your card <CARD:card> was declined at <TEXT:text>?
t00013 (226 msgs): salary of <CURRENCY:currency> <AMOUNT:salary> credited to account <CARD:account> on <DATE:date>
t00043 (222 msgs): عزيزنا العميل تم <TEXT:العميل>?
t00025 (218 msgs): your verification code is <NUM:code> . login from ip <REF:ip> at <TIME:time> <TEXT:login>?
t00010 (216 msgs): رمز التحقق الخاص بك هو <NUM:الخاص> صالح لمده <NUM:لمده> دقايق لا تشاركه مع احد
t00066 (211 msgs): transfer of <CURRENCY:currency> <AMOUNT:transfer> to <TEXT:text>?
t00015 (208 msgs): purchase of <CURRENCY:currency> <AMOUNT:purchase> at <TEXT:text> on card ending <CARD:ending> on <DATETIME:datetime> . available balance <CURRENCY:currency_2> <AMOUNT:balance>
t00038 (202 msgs): طلبيتك <NUM:طلبيتك> في الطريق الي <TEXT:الي>?
t00109 (202 msgs): alert host <TEXT:host>
t00016 (197 msgs): your order <NUM:order> has be

## 4. Benchmark production matching

In [8]:
matcher = TemplateMatcher(library)

# Warm up the matcher once (JIT compilation of regexes happens on first use)
_ = [matcher.match(m) for m in messages[:10]]

# Benchmark: repeat the corpus enough times to get a stable number
repeats = max(3, 10000 // len(messages))
batch = messages * repeats
t0 = time.perf_counter()
for m in batch:
    matcher.match(m)
bench_seconds = time.perf_counter() - t0

print(f"Messages: {len(batch):,}")
print(f"Time: {bench_seconds:.3f}s")
print(f"Throughput: {len(batch) / bench_seconds:,.0f} msg/s")


Messages: 15,000
Time: 1.808s
Throughput: 8,296 msg/s


## 5. Profile learning

In [9]:
cProfile.runctx(
    "learn_templates(messages, config=config)",
    globals(),
    locals(),
    filename="/tmp/learn.prof",
)
p = pstats.Stats("/tmp/learn.prof")
p.strip_dirs().sort_stats("cumulative").print_stats(20)


Sun Aug 23 05:05:43 2026    /tmp/learn.prof

         3141832 function calls in 1.405 seconds

   Ordered by: cumulative time
   List reduced from 162 to 20 due to restriction <20>

   ncalls  tottime  percall  cumtime  percall filename:lineno(function)
        1    0.000    0.000    1.494    1.494 {built-in method builtins.exec}
        1    0.000    0.000    1.494    1.494 <string>:1(<module>)
        1    0.003    0.003    1.494    1.494 learn.py:432(learn_templates)
        1    0.000    0.000    1.491    1.491 learn.py:102(fit)
        1    0.014    0.014    1.292    1.292 learn.py:90(fit_partial)
     5000    0.015    0.000    1.082    0.000 learn.py:64(prepare)
     5000    0.159    0.000    0.692    0.000 entities.py:166(mask_tokens)
    68976    0.168    0.000    0.400    0.000 entities.py:156(_currency_run)
   254415    0.149    0.000    0.240    0.000 {method 'join' of 'str' objects}
     5000    0.137    0.000    0.214    0.000 tokenizer.py:38(tokenize)
        1    0.001  

## 6. Profile matching

In [10]:
cProfile.runctx(
    "for m in messages: matcher.match(m)",
    globals(),
    locals(),
    filename="/tmp/match.prof",
)
p = pstats.Stats("/tmp/match.prof")
p.strip_dirs().sort_stats("cumulative").print_stats(20)


Sun Aug 23 05:05:44 2026    /tmp/match.prof

         2309297 function calls in 1.089 seconds

   Ordered by: cumulative time
   List reduced from 50 to 20 due to restriction <20>

   ncalls  tottime  percall  cumtime  percall filename:lineno(function)
        1    0.000    0.000    1.175    1.175 {built-in method builtins.exec}
        1    0.005    0.005    1.175    1.175 <string>:1(<module>)
     5000    0.008    0.000    1.170    0.000 matcher.py:122(match)
     5000    0.013    0.000    1.034    0.000 learn.py:64(prepare)
     5000    0.154    0.000    0.659    0.000 entities.py:166(mask_tokens)
    68976    0.157    0.000    0.382    0.000 entities.py:156(_currency_run)
   268559    0.144    0.000    0.232    0.000 {method 'join' of 'str' objects}
     5000    0.129    0.000    0.205    0.000 tokenizer.py:38(tokenize)
     5000    0.021    0.000    0.130    0.000 normalize.py:133(normalize)
     5000    0.007    0.000    0.129    0.000 matcher.py:126(match_canonical)
     5000   

## 7. Sample matches

In [11]:
print("Random sample matches:\n")
for m in random.sample(messages, min(10, len(messages))):
    r = matcher.match(m)
    print(m)
    if r is None:
        print("  -> no match\n")
    else:
        print("  ->", r.template_id, r.entities, "\n")


Random sample matches:

Your verification code is 337672. Login from ip 10.29.12.108 at 07:07
  -> t00025 {'code': '337672', 'ip': '10.29.12.108', 'time': '07:07'} 

Transfer of ر.س 14,846.70 to خالد العتيبي completed. Ref 980028929. Balance ر.س 4,371.58
  -> t00066 {'currency': 'ر.س', 'transfer': '14,846.70', 'text': 'خالد العتيبي completed . ref 980028929 . balance ر . س 4,371.58'} 

ALERT host srv-prod-07 cpu 54% at 07:53 service الحوالات الفورية status cancelled
  -> t00109 {'host': 'srv - prod - 07 cpu 54 % at 07:53 service الحوالات الفوريه status cancelled'} 

Your OTP is 717751. Valid for 10 minutes. Do not share it with anyone
  -> t00009 {'otp': '717751', 'valid': '10'} 

Bill payment ر.س 3,734.11 to نون succeeded. Fee ر.س 74.06. Ref 472206794
  -> t00012 {'currency': 'ر.س', 'payment': '3,734.11', 'text': 'نون', 'currency_2': 'ر.س', 'fee': '74.06', 'ref': '472206794'} 

عزيزنا العميل تم تفعيل خدمة المدفوعات الدولية على حسابك ***317 للاستفسار اتصل على 966527247110
  -> t00043 {

## 8. Full markdown report

In [12]:
report = render_library(library, {**cov, "learn_seconds": learn_seconds, "messages": len(messages)})
print(report)


# Template library

- templates: **66**
- messages: **5000**
- messages_scored: **5000**
- coverage: **0.964**
- learn_seconds: **0.825**

## Templates

### `t00002` (240 msgs)

```
عمليه شراء بمبلغ <AMOUNT:بمبلغ> <CURRENCY:currency> لدي <TEXT:text>?
```

| slot | entity | distinct | examples |
| --- | --- | --- | --- |
| بمبلغ | AMOUNT | 0 |  |
| currency | CURRENCY | 0 |  |
| text | TEXT | 0 |  |

example: `عمليه شراء بمبلغ 82,513.88 ر . س لدي بنده بالبطاقه xxxx 3243 بتاريخ 2024-05-24 الرصيد المتاح 248,278.52`

### `t00044` (227 msgs)

```
your card <CARD:card> was declined at <TEXT:text>?
```

| slot | entity | distinct | examples |
| --- | --- | --- | --- |
| card | CARD | 0 |  |
| text | TEXT | 0 |  |

example: `your card * * * * 1375 was declined at نون due to insufficient funds`

### `t00013` (226 msgs)

```
salary of <CURRENCY:currency> <AMOUNT:salary> credited to account <CARD:account> on <DATE:date>
```

| slot | entity | distinct | examples |
| --- | --- | --- | --- |
| curr

## Next steps

- Replace `CSV_PATH` with your real CSV and rerun all cells.
- Tune `SIM` (similarity threshold) if coverage is too low or templates too noisy.
- Use `LEXICON = "finance"` (or a JSON file) to rename generic slots like `text`
  into domain names like `merchant` / `beneficiary`.
- Persist `model.json` and use `patgen match` or `TemplateMatcher` in production.